# Video-to-gaze pipeline

This notebook generates saliency latents for one input video, runs the bundled `final_model_90_45/checkpoint_70.pth`, and renders its predicted gaze trajectory over the source video. Run the cells from the repository root. All generated artifacts are written under `C:\\Users\\jk8659\\NYU\\research\\Scanpath\\diffeye\\artifacts`.

In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd().resolve()
assert (REPO / 'generate_unisal_latents.py').exists(), 'Open this notebook from the repository root.'

# Change this to the video you want to process.
ARTIFACT_ROOT = Path(r"C:\Users\jk8659\NYU\research\Scanpath\diffeye\artifacts")
INPUT_VIDEO = ARTIFACT_ROOT / 'datasets' / 'DHF1K' / 'eval_videos' / '125.AVI'
assert INPUT_VIDEO.exists(), f'Video not found: {INPUT_VIDEO}'

PIPELINE_ROOT = ARTIFACT_ROOT / 'full_video_pipeline'
LATENT_ROOT = PIPELINE_ROOT / 'saliency_latents'
SAMPLE_ROOT = PIPELINE_ROOT / 'gaze_samples'
CHECKPOINT = REPO / 'final_model_90_45' / 'checkpoint_70.pth'
CONFIG = REPO / 'final_model_90_45' / 'inference_config.yaml'

assert CHECKPOINT.exists(), f'Checkpoint not found: {CHECKPOINT}'
assert CONFIG.exists(), f'Config not found: {CONFIG}'
print(f'Input: {INPUT_VIDEO}')
print(f'Outputs: {PIPELINE_ROOT}')

In [ ]:
# Generate saliency latents from the input video in this kernel.
from contextlib import contextmanager
import generate_unisal_latents

@contextmanager
def temporary_argv(args):
    original_argv = sys.argv
    sys.argv = [original_argv[0], *map(str, args)]
    try:
        yield
    finally:
        sys.argv = original_argv

CONDITIONING_DIR = LATENT_ROOT / INPUT_VIDEO.stem
with temporary_argv(['--video-path', INPUT_VIDEO, '--output-root', LATENT_ROOT]):
    generate_unisal_latents.main()

latent_files = sorted(CONDITIONING_DIR.glob('*.pt'))
assert latent_files, f'No latents were created in {CONDITIONING_DIR}'
print(f'Using {len(latent_files)} saliency latents from {CONDITIONING_DIR}')

In [ ]:
# Generate one deterministic gaze trajectory and render it as an overlay video.
import sample_video

with temporary_argv([
    '--config', CONFIG,
    '--checkpoint', CHECKPOINT,
    '--video-path', INPUT_VIDEO,
    '--conditioning-dir', CONDITIONING_DIR,
    '--output-dir', SAMPLE_ROOT,
    '--num-samples', '1',
    '--seed', '12',
    '--overlay-trail', '20',
]):
    sample_video.main()

In [ ]:
from IPython.display import Video, display

OUTPUT_DIR = SAMPLE_ROOT / INPUT_VIDEO.stem / 'sample_000'
OVERLAY_VIDEO = OUTPUT_DIR / 'overlay.mp4'
SCANPATH_CSV = OUTPUT_DIR / 'scanpath.csv'

assert OVERLAY_VIDEO.exists(), f'Overlay was not created: {OVERLAY_VIDEO}'
print(f'Overlay video: {OVERLAY_VIDEO}')
print(f'Gaze coordinates: {SCANPATH_CSV}')
